Tanner/trapping-set structural sections now live in [the trapping-set workflow](../../trapping_sets/README.md). This entry owns acquisition, belief/prior/scaling diagnostics and threshold sweeps.


# Low Weight Failure Analysis
Configuration: `config`. Figures remain local; datasets are under `experiment/data/`.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
low_weight_failure_analysis_experiment_dir = ROOT / 'experiment/research/low_weight_failures'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

low_weight_failure_analysis_data_paths = ExperimentPaths(ROOT, low_weight_failure_analysis_experiment_dir / 'config/paths.json')

import json

low_weight_failure_analysis_config = json.loads((low_weight_failure_analysis_experiment_dir / 'config/runtime.json').read_text())
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.


In [ ]:
import multiprocessing
import sys
import os
from pathlib import Path
from typing import Literal
import numpy as np
import pandas as pd
import sinter
import stim
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product, combinations

low_weight_failure_analysis_data_dir = low_weight_failure_analysis_data_paths.run()
low_weight_failure_analysis_data_dir.mkdir(parents=True, exist_ok=True)

from utils.misc import *

%load_ext autoreload
%autoreload 2


In [ ]:
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders import BPOSD
from src.decoders import Tesseract


In [ ]:
low_weight_failure_analysis_ca_codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5()}

from experiment.utils import decoder_setup

def low_weight_failure_analysis_build_decoder_pair(code, p):
    return decoder_setup.build_decoder_pair(code, p, bposd_type=BPOSD, tesseract_type=Tesseract)


In [ ]:
import multiprocessing
import sys
import os
from pathlib import Path
from typing import Literal
import numpy as np
import pandas as pd
import sinter
import stim
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product, combinations
from utils.misc import *
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders import BPOSD
from src.decoders import Tesseract
from utils.graph_vis import visualize_graph_plotly, flatten_graph
from utils.graph_struct import build_tanner_graph_from_dem, graph_struct
from beliefmatching import detector_error_model_to_check_matrices as low_weight_failure_analysis_detector_error_model_to_check_matrices
from utils.sample import count_samples_union
from matplotlib_venn import venn2
from utils.sample import collect_samples_with_errors
from src.decoders import CythonBPOSD
from utils.belief_tracking import plot_belief_tracking, plot_history_heatmap
from utils.graph_struct import build_tanner_graph_from_dem
from utils.graph_vis import animate_hard_decisions_on_graph, visualize_det_on_tanner_graph, visualize_graph_plotly


In [ ]:
import numpy as np

low_weight_failure_analysis_noise_model = 'circuit_level'
low_weight_failure_analysis_d_rounds = 1
low_weight_failure_analysis_max_shots = 1000000000
max_errors = 2000
low_weight_failure_analysis_p_values = np.geomspace(1e-08, 0.1, 12)


### Prepare the acquisition inputs

Reconstruct the selected circuit/DEM and read the historical failure snapshot that supplies acquisition settings. The following sections generate reusable fixed-weight support tables and then inspect selected cases.


In [ ]:
low_weight_failure_analysis_error_rate = ErrorRate(p=0.01, p_meas=0.0)
low_weight_failure_analysis_builder = CircuitBuilder(low_weight_failure_analysis_ca_codes['CA_9_2_5'])
low_weight_failure_analysis_circ = low_weight_failure_analysis_builder.get_circuit_level_circuit(low_weight_failure_analysis_error_rate, d_rounds=1)
low_weight_failure_analysis_dem = low_weight_failure_analysis_circ.detector_error_model()
TG, low_weight_failure_analysis_ = build_tanner_graph_from_dem(low_weight_failure_analysis_dem)


In [ ]:
low_weight_failure_analysis_bfts_cases = pd.read_csv(low_weight_failure_analysis_data_paths.input('bfts_cases_csv'), dtype={
    'det': str,
    'err': str,
    'true': str,
    'pred_b': str,
    'pred_t': str,
})


# Low Hamming Weight Error

In [ ]:
import networkx as nx
from beliefmatching import detector_error_model_to_check_matrices as low_weight_failure_analysis_detector_error_model_to_check_matrices
from utils.sample import sample_on_error_vec
from utils.misc import pack_bits_little as low_weight_failure_analysis_pack_bits_little
from utils.sample import packed_rows_equal
from utils.graph_struct import build_tanner_graph_from_dem, error_llr_weight, get_error_time, error_cross_time, error_diameter, dem_error_to_explanation, most_occurring_circ_error
from utils.graph_vis import visualize_graph_plotly, visualize_det_on_tanner_graph, animate_hard_decisions_on_graph


In [ ]:
low_weight_failure_analysis_weight_dir = low_weight_failure_analysis_data_paths.output('error_weight')
os.makedirs(low_weight_failure_analysis_weight_dir, exist_ok=True)
low_weight_failure_analysis_num_errs = 33
low_weight_failure_analysis_num_dets = 14
low_weight_failure_analysis_num_obs = 2
low_weight_failure_analysis_max_iter = 60
low_weight_failure_analysis_code = low_weight_failure_analysis_ca_codes['CA_9_2_5']


In [ ]:
low_weight_failure_analysis_p_values = low_weight_failure_analysis_bfts_cases['p'].unique()
low_weight_failure_analysis_p_values


## Weight-2

### Enumerate and decode weight-2 supports

Build the weight-2 inputs, decode them with the configured references and reload the saved table for analysis. Keep support-based failure fractions separate from physical-noise LER.


In [ ]:
low_weight_failure_analysis_weight = 2


In [ ]:
low_weight_failure_analysis_dem = CircuitBuilder(low_weight_failure_analysis_code).get_circuit_level_circuit(ErrorRate(p=0.01, p_meas=0.0), d_rounds=1).detector_error_model()
low_weight_failure_analysis_demMatrices = low_weight_failure_analysis_detector_error_model_to_check_matrices(low_weight_failure_analysis_dem, allow_undecomposed_hyperedges=True)
low_weight_failure_analysis_h = low_weight_failure_analysis_demMatrices.check_matrix
low_weight_failure_analysis_l = low_weight_failure_analysis_demMatrices.observables_matrix
low_weight_failure_analysis_errs, low_weight_failure_analysis_dets, low_weight_failure_analysis_obs = sample_on_error_vec(low_weight_failure_analysis_num_errs, low_weight_failure_analysis_weight, low_weight_failure_analysis_h, low_weight_failure_analysis_l)
low_weight_failure_analysis_dets_packed = low_weight_failure_analysis_pack_bits_little(low_weight_failure_analysis_dets)
low_weight_failure_analysis_obs_packed = low_weight_failure_analysis_pack_bits_little(low_weight_failure_analysis_obs)


In [ ]:
from tqdm.auto import tqdm

low_weight_failure_analysis_recs = []
low_weight_failure_analysis_builder = CircuitBuilder(low_weight_failure_analysis_code)
low_weight_failure_analysis_tag_builder = CircuitBuilder(low_weight_failure_analysis_code, need_tags=True)

for low_weight_failure_analysis_p in tqdm(low_weight_failure_analysis_p_values, desc='Scientific sweep', disable=low_weight_failure_analysis_config['print_level'] == 1):
    low_weight_failure_analysis_error_rate = ErrorRate(p=float(low_weight_failure_analysis_p), p_meas=0.0)
    low_weight_failure_analysis_circ = low_weight_failure_analysis_builder.get_circuit_level_circuit(low_weight_failure_analysis_error_rate, d_rounds=1)
    low_weight_failure_analysis_tag_circ = low_weight_failure_analysis_tag_builder.get_circuit_level_circuit(low_weight_failure_analysis_error_rate, d_rounds=1)
    low_weight_failure_analysis_dem = low_weight_failure_analysis_circ.detector_error_model()
    low_weight_failure_analysis_g, low_weight_failure_analysis_ = build_tanner_graph_from_dem(low_weight_failure_analysis_dem)

    low_weight_failure_analysis_compiled_b = BPOSD(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, low_weight_failure_analysis_max_iter).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)
    low_weight_failure_analysis_compiled_t = Tesseract(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)

    low_weight_failure_analysis_corr_b, low_weight_failure_analysis_pred_b = low_weight_failure_analysis_compiled_b.decode_with_correction_shots_bit_packed(bit_packed_detection_event_data=low_weight_failure_analysis_dets_packed)
    low_weight_failure_analysis_corr_b = unpack_bits_little(low_weight_failure_analysis_corr_b, low_weight_failure_analysis_num_errs)
    low_weight_failure_analysis_pred_b = unpack_bits_little(low_weight_failure_analysis_pred_b, low_weight_failure_analysis_num_obs)

    low_weight_failure_analysis_corr_t = np.zeros_like(low_weight_failure_analysis_corr_b)
    for low_weight_failure_analysis_i, low_weight_failure_analysis_det in enumerate(low_weight_failure_analysis_dets):
        low_weight_failure_analysis_corr_t[low_weight_failure_analysis_i, :] = id2bits(low_weight_failure_analysis_compiled_t.decoder.decode_to_errors(low_weight_failure_analysis_det), low_weight_failure_analysis_num_errs)

    low_weight_failure_analysis_pred_t = low_weight_failure_analysis_corr_t @ low_weight_failure_analysis_l.T % 2

    low_weight_failure_analysis_recs.extend(
        {
            'p': low_weight_failure_analysis_p,
            'err': bits2str(e),
            'err_cross_time': error_cross_time(get_error_time(low_weight_failure_analysis_dem, bits2str(e))),
            'err_llr_weight': error_llr_weight(low_weight_failure_analysis_dem, bits2str(e)),
            'err_diameter': error_diameter(low_weight_failure_analysis_g, bits2str(e)),
            'err_most_common_src': [most_occurring_circ_error(dem_error_to_explanation(low_weight_failure_analysis_tag_circ, low_weight_failure_analysis_dem), err_idx) for err_idx in bits2id(e)],
            'det': bits2str(d),
            'corr_b': bits2str(cb),
            'corr_b_cross_time': error_cross_time(get_error_time(low_weight_failure_analysis_dem, bits2str(cb))),
            'corr_b_llr_weight': error_llr_weight(low_weight_failure_analysis_dem, bits2str(cb)),
            'corr_b_diameter': error_diameter(low_weight_failure_analysis_g, bits2str(cb)),
            'bp_converged': low_weight_failure_analysis_compiled_b.decoder.converge(),
            'corr_t': bits2str(ct),
            'corr_t_cross_time': error_cross_time(get_error_time(low_weight_failure_analysis_dem, bits2str(ct))),
            'corr_t_llr_weight': error_llr_weight(low_weight_failure_analysis_dem, bits2str(ct)),
            'corr_t_diameter': error_diameter(low_weight_failure_analysis_g, bits2str(ct)),
            'pred_b': bits2str(pb),
            'pred_t': bits2str(pt),
            'true': bits2str(ob),
        }
        for (e, d, cb, ct, pb, pt, ob) in zip(
            low_weight_failure_analysis_errs,
            low_weight_failure_analysis_dets,
            low_weight_failure_analysis_corr_b,
            low_weight_failure_analysis_corr_t,
            low_weight_failure_analysis_pred_b,
            low_weight_failure_analysis_pred_t,
            low_weight_failure_analysis_obs,
        )
    )

low_weight_failure_analysis_weight2_data = pd.DataFrame(low_weight_failure_analysis_recs)
low_weight_failure_analysis_save_path = low_weight_failure_analysis_data_paths.output('weight__low_weight_failure_analysis_weight__data_csv', low_weight_failure_analysis_weight=low_weight_failure_analysis_weight)

low_weight_failure_analysis_weight2_data.to_csv(low_weight_failure_analysis_save_path, index=False)
if low_weight_failure_analysis_config['print_level'] >= 2:
    print(f'saved: {low_weight_failure_analysis_save_path}')


In [ ]:
low_weight_failure_analysis_types = {
    'p': float,
    'err': str,
    'det': str,
    'corr_b': str,
    'corr_t': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

low_weight_failure_analysis_weight2_data = pd.read_csv(low_weight_failure_analysis_data_paths.output('weight__low_weight_failure_analysis_weight__data_csv', low_weight_failure_analysis_weight=low_weight_failure_analysis_weight), dtype=low_weight_failure_analysis_types)


In [ ]:
low_weight_failure_analysis_mask = (low_weight_failure_analysis_weight2_data['pred_b'] != low_weight_failure_analysis_weight2_data['true']) & (low_weight_failure_analysis_weight2_data['pred_t'] == low_weight_failure_analysis_weight2_data['true'])
low_weight_failure_analysis_filtered_data = low_weight_failure_analysis_weight2_data[low_weight_failure_analysis_mask]
low_weight_failure_analysis_filtered_data


### LLR Weight and Diameter

In [ ]:
low_weight_failure_analysis_fig, low_weight_failure_analysis_ax = plt.subplots(2, 1, figsize=(18, 10))
low_weight_failure_analysis_ax = low_weight_failure_analysis_ax.flatten()
low_weight_failure_analysis_df = low_weight_failure_analysis_filtered_data[['err_llr_weight', 'corr_b_llr_weight', 'corr_t_llr_weight']].copy().reset_index(drop=True)

low_weight_failure_analysis_df.plot(
    figsize=(12, 5),
    marker='o',
    ax=low_weight_failure_analysis_ax[0],
    xlabel='case index',
    ylabel='LLR weight',
    title='LLR weights for each case',
)
low_weight_failure_analysis_df = low_weight_failure_analysis_filtered_data[['err_diameter', 'corr_b_diameter', 'corr_t_diameter']].copy().reset_index(drop=True)
low_weight_failure_analysis_df.plot(
    figsize=(12, 5),
    marker='o',
    ax=low_weight_failure_analysis_ax[1],
    xlabel='case index',
    ylabel='Diameter',
    title='Diameters for each case',
)
plt.tight_layout()

plt.show()


### Belief Tracking

In [ ]:
from utils.belief_tracking import analyze_belief_history, plot_belief_analysis


In [ ]:
low_weight_failure_analysis_filtered_data.groupby(['det', 'err']).value_counts()


#### 1

### First weight-2 trajectory

Trace the selected detector/error pair, inspect its belief and hard-decision history and compare the standard correction with forced OSD. Logical correctness is evaluated separately from exact recovery of the sampled support.


In [ ]:
row = low_weight_failure_analysis_filtered_data[(low_weight_failure_analysis_filtered_data['det'] == '11010000000010') & (low_weight_failure_analysis_filtered_data['err'] == '001000000100000000000000000000000')].iloc[-1]
low_weight_failure_analysis_error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
low_weight_failure_analysis_builder = CircuitBuilder(low_weight_failure_analysis_code)
low_weight_failure_analysis_circ = low_weight_failure_analysis_builder.get_circuit_level_circuit(low_weight_failure_analysis_error_rate, d_rounds=1)
low_weight_failure_analysis_dem = low_weight_failure_analysis_circ.detector_error_model()
low_weight_failure_analysis_compiled_b = CythonBPOSD(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, low_weight_failure_analysis_max_iter).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)
low_weight_failure_analysis_compiled_t = Tesseract(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)


In [ ]:
low_weight_failure_analysis_pred_b = low_weight_failure_analysis_compiled_b.decoder.decode(str2bits(row['det']))
low_weight_failure_analysis_belief_history = low_weight_failure_analysis_compiled_b.decoder.belief_history.copy()
low_weight_failure_analysis_pred_t = low_weight_failure_analysis_compiled_t.decoder.decode(str2bits(row['det']))
low_weight_failure_analysis_info_b = {
    'pred_b': low_weight_failure_analysis_pred_b,
    'belief_history_shape': low_weight_failure_analysis_belief_history.shape,
    'is_bp_converged': low_weight_failure_analysis_compiled_b.decoder.is_bp_converged,
    'is_osd_satisfied': low_weight_failure_analysis_compiled_b.decoder.is_osd_satisfied,
}
low_weight_failure_analysis_info_t = {'pred_t': low_weight_failure_analysis_pred_t.astype(int)}

if low_weight_failure_analysis_config['print_level'] >= 2:
    print('BPOSD info:', low_weight_failure_analysis_info_b)
if low_weight_failure_analysis_config['print_level'] >= 2:
    print('Tesseract info:', low_weight_failure_analysis_info_t)


In [ ]:
nodes = set(range(33)) - set([0, 1, 2, 7, 9, 13, 25, 31])
nodes = np.array(list(nodes))


In [ ]:
low_weight_failure_analysis_fig, low_weight_failure_analysis_ax = plt.subplots(2, 1, figsize=(12, 12))
plot_belief_tracking(low_weight_failure_analysis_belief_history[:, np.array([0, 1, 2, 7, 9, 13, 25, 31])], low_weight_failure_analysis_ax[0])
plot_belief_tracking(low_weight_failure_analysis_belief_history[:, nodes], low_weight_failure_analysis_ax[1])


In [ ]:
summary_df, variable_df, info = analyze_belief_history(low_weight_failure_analysis_belief_history, low_weight_failure_analysis_max_iter, low_weight_failure_analysis_compiled_b.decoder.H, str2bits(row['det']))
summary_df


In [ ]:
low_weight_failure_analysis_fig, low_weight_failure_analysis_ax = plot_belief_analysis(info, variable_df)


In [ ]:
str2id('001000000100000000000000000000000', 14)


In [ ]:
TG, low_weight_failure_analysis_ = build_tanner_graph_from_dem(low_weight_failure_analysis_dem)
visualize_det_on_tanner_graph(TG, '11010000000010', other_events={('oscillation', 'green', 'circle'): id2str([0, 1, 2, 7, 9, 13, 25, 31], 33)})


In [ ]:
low_weight_failure_analysis_corr_b = low_weight_failure_analysis_compiled_b.decoder.decode_to_correction(str2bits(row['det']))
bits2id(low_weight_failure_analysis_corr_b, 14)


In [ ]:
low_weight_failure_analysis_corr_b = low_weight_failure_analysis_compiled_b.decoder.decode_to_correction(str2bits(row['det']), force_osd=True)
bits2id(low_weight_failure_analysis_corr_b, 14)


#### 2

### Alternative support for the same syndrome

Repeat the trace for another recorded error explanation of the same detector pattern. This tests what changes when the physical explanation differs even though the observed syndrome is shared.


In [ ]:
row = low_weight_failure_analysis_filtered_data[(low_weight_failure_analysis_filtered_data['det'] == '11010000000010') & (low_weight_failure_analysis_filtered_data['err'] == '010000010000000000000000000000000')].iloc[-1]
low_weight_failure_analysis_error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
low_weight_failure_analysis_builder = CircuitBuilder(low_weight_failure_analysis_code)
low_weight_failure_analysis_circ = low_weight_failure_analysis_builder.get_circuit_level_circuit(low_weight_failure_analysis_error_rate, d_rounds=1)
low_weight_failure_analysis_dem = low_weight_failure_analysis_circ.detector_error_model()
low_weight_failure_analysis_compiled_b = CythonBPOSD(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, low_weight_failure_analysis_max_iter).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)
low_weight_failure_analysis_compiled_t = Tesseract(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)


In [ ]:
low_weight_failure_analysis_pred_b = low_weight_failure_analysis_compiled_b.decoder.decode(str2bits(row['det']))
low_weight_failure_analysis_belief_history = low_weight_failure_analysis_compiled_b.decoder.belief_history.copy()
low_weight_failure_analysis_pred_t = low_weight_failure_analysis_compiled_t.decoder.decode(str2bits(row['det']))
low_weight_failure_analysis_info_b = {
    'pred_b': low_weight_failure_analysis_pred_b,
    'belief_history_shape': low_weight_failure_analysis_belief_history.shape,
    'is_bp_converged': low_weight_failure_analysis_compiled_b.decoder.is_bp_converged,
    'is_osd_satisfied': low_weight_failure_analysis_compiled_b.decoder.is_osd_satisfied,
}
low_weight_failure_analysis_info_t = {'pred_t': low_weight_failure_analysis_pred_t.astype(int)}

if low_weight_failure_analysis_config['print_level'] >= 2:
    print('BPOSD info:', low_weight_failure_analysis_info_b)
if low_weight_failure_analysis_config['print_level'] >= 2:
    print('Tesseract info:', low_weight_failure_analysis_info_t)


In [ ]:
summary_df, variable_df, info = analyze_belief_history(low_weight_failure_analysis_belief_history, low_weight_failure_analysis_max_iter, low_weight_failure_analysis_compiled_b.decoder.H, str2bits(row['det']))
summary_df


In [ ]:
low_weight_failure_analysis_fig, low_weight_failure_analysis_ax = plot_belief_analysis(info, variable_df)


In [ ]:
low_weight_failure_analysis_corr_b = low_weight_failure_analysis_compiled_b.decoder.decode_to_correction(str2bits(row['det']))
bits2id(low_weight_failure_analysis_corr_b, 14)


In [ ]:
low_weight_failure_analysis_corr_b = low_weight_failure_analysis_compiled_b.decoder.decode_to_correction(str2bits(row['det']), force_osd=True)
bits2id(low_weight_failure_analysis_corr_b, 14)


#### 3

### Second syndrome and paired explanations

The next two examples inspect another detector pattern and its alternative supports using the same belief-analysis procedure. Compare the actual trajectories rather than assuming symmetry of the drawn error pattern implies equivalent decoding.


In [ ]:
row = low_weight_failure_analysis_filtered_data[(low_weight_failure_analysis_filtered_data['det'] == '11011000000110') & (low_weight_failure_analysis_filtered_data['err'] == '001000010000000000000000000000000')].iloc[-1]
low_weight_failure_analysis_error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
low_weight_failure_analysis_builder = CircuitBuilder(low_weight_failure_analysis_code)
low_weight_failure_analysis_circ = low_weight_failure_analysis_builder.get_circuit_level_circuit(low_weight_failure_analysis_error_rate, d_rounds=1)
low_weight_failure_analysis_dem = low_weight_failure_analysis_circ.detector_error_model()
low_weight_failure_analysis_compiled_b = CythonBPOSD(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, low_weight_failure_analysis_max_iter).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)
low_weight_failure_analysis_compiled_t = Tesseract(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)


In [ ]:
low_weight_failure_analysis_pred_b = low_weight_failure_analysis_compiled_b.decoder.decode(str2bits(row['det']))
low_weight_failure_analysis_belief_history = low_weight_failure_analysis_compiled_b.decoder.belief_history.copy()
low_weight_failure_analysis_pred_t = low_weight_failure_analysis_compiled_t.decoder.decode(str2bits(row['det']))
low_weight_failure_analysis_info_b = {
    'pred_b': low_weight_failure_analysis_pred_b,
    'belief_history_shape': low_weight_failure_analysis_belief_history.shape,
    'is_bp_converged': low_weight_failure_analysis_compiled_b.decoder.is_bp_converged,
    'is_osd_satisfied': low_weight_failure_analysis_compiled_b.decoder.is_osd_satisfied,
}
low_weight_failure_analysis_info_t = {'pred_t': low_weight_failure_analysis_pred_t.astype(int)}

if low_weight_failure_analysis_config['print_level'] >= 2:
    print('BPOSD info:', low_weight_failure_analysis_info_b)
if low_weight_failure_analysis_config['print_level'] >= 2:
    print('Tesseract info:', low_weight_failure_analysis_info_t)


In [ ]:
summary_df, variable_df, info = analyze_belief_history(low_weight_failure_analysis_belief_history, low_weight_failure_analysis_max_iter, low_weight_failure_analysis_compiled_b.decoder.H, str2bits(row['det']))
summary_df


In [ ]:
low_weight_failure_analysis_fig, low_weight_failure_analysis_ax = plot_belief_analysis(info, variable_df)


In [ ]:
low_weight_failure_analysis_corr_b = low_weight_failure_analysis_compiled_b.decoder.decode_to_correction(str2bits(row['det']))
bits2id(low_weight_failure_analysis_corr_b, 14)


In [ ]:
low_weight_failure_analysis_corr_b = low_weight_failure_analysis_compiled_b.decoder.decode_to_correction(str2bits(row['det']), force_osd=True)
bits2id(low_weight_failure_analysis_corr_b, 14)


#### 4

In [ ]:
row = low_weight_failure_analysis_filtered_data[(low_weight_failure_analysis_filtered_data['det'] == '11011000000110') & (low_weight_failure_analysis_filtered_data['err'] == '010000000100000000000000000000000')].iloc[-1]
low_weight_failure_analysis_error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
low_weight_failure_analysis_builder = CircuitBuilder(low_weight_failure_analysis_code)
low_weight_failure_analysis_circ = low_weight_failure_analysis_builder.get_circuit_level_circuit(low_weight_failure_analysis_error_rate, d_rounds=1)
low_weight_failure_analysis_dem = low_weight_failure_analysis_circ.detector_error_model()
low_weight_failure_analysis_compiled_b = CythonBPOSD(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, low_weight_failure_analysis_max_iter).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)
low_weight_failure_analysis_compiled_t = Tesseract(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)


In [ ]:
low_weight_failure_analysis_pred_b = low_weight_failure_analysis_compiled_b.decoder.decode(str2bits(row['det']))
low_weight_failure_analysis_belief_history = low_weight_failure_analysis_compiled_b.decoder.belief_history.copy()
low_weight_failure_analysis_pred_t = low_weight_failure_analysis_compiled_t.decoder.decode(str2bits(row['det']))
low_weight_failure_analysis_info_b = {
    'pred_b': low_weight_failure_analysis_pred_b,
    'belief_history_shape': low_weight_failure_analysis_belief_history.shape,
    'is_bp_converged': low_weight_failure_analysis_compiled_b.decoder.is_bp_converged,
    'is_osd_satisfied': low_weight_failure_analysis_compiled_b.decoder.is_osd_satisfied,
}
low_weight_failure_analysis_info_t = {'pred_t': low_weight_failure_analysis_pred_t.astype(int)}

if low_weight_failure_analysis_config['print_level'] >= 2:
    print('BPOSD info:', low_weight_failure_analysis_info_b)
if low_weight_failure_analysis_config['print_level'] >= 2:
    print('Tesseract info:', low_weight_failure_analysis_info_t)


In [ ]:
summary_df, variable_df, info = analyze_belief_history(low_weight_failure_analysis_belief_history, low_weight_failure_analysis_max_iter, low_weight_failure_analysis_compiled_b.decoder.H, str2bits(row['det']))
summary_df


In [ ]:
low_weight_failure_analysis_fig, low_weight_failure_analysis_ax = plot_belief_analysis(info, variable_df)


In [ ]:
TG, low_weight_failure_analysis_ = build_tanner_graph_from_dem(low_weight_failure_analysis_dem)
visualize_det_on_tanner_graph(TG, '11011000000110', other_events={('oscillation', 'green', 'circle'): id2str([0, 1, 2, 7, 9, 13, 14, 17], 33)})


In [ ]:
low_weight_failure_analysis_corr_b = low_weight_failure_analysis_compiled_b.decoder.decode_to_correction(str2bits(row['det']))
bits2id(low_weight_failure_analysis_corr_b, 14)


In [ ]:
low_weight_failure_analysis_corr_b = low_weight_failure_analysis_compiled_b.decoder.decode_to_correction(str2bits(row['det']), force_osd=True)
bits2id(low_weight_failure_analysis_corr_b, 14)


### Error Explanation

### Relate DEM faults to circuit operations

Inspect a recorded support and ask the circuit to explain selected DEM terms. This connects graph-level error variables with concrete fault locations in the circuit.


In [ ]:
row = low_weight_failure_analysis_filtered_data.loc[2179]
low_weight_failure_analysis_builder = CircuitBuilder(low_weight_failure_analysis_code)
low_weight_failure_analysis_circ = low_weight_failure_analysis_builder.get_circuit_level_circuit(ErrorRate(p=float(row['p']), p_meas=0.0), d_rounds=1)
low_weight_failure_analysis_dem = low_weight_failure_analysis_circ.detector_error_model()
TG, low_weight_failure_analysis_ = build_tanner_graph_from_dem(low_weight_failure_analysis_dem)


In [ ]:
err_ids = str2id(row['err'], low_weight_failure_analysis_dem.num_detectors)
corr_b_ids = str2id(row['corr_b'], low_weight_failure_analysis_dem.num_detectors)
(err_ids, corr_b_ids)


In [ ]:
visualize_det_on_tanner_graph(TG, row['det'], other_events={
    ('error', 'green', 'circle'): row['err'],
    ('bposd_corr', 'orange', 'diamond'): row['corr_b'],
})


In [ ]:
explained_errs = low_weight_failure_analysis_circ.explain_detector_error_model_errors(dem_filter=stim.DetectorErrorModel('error(0.0123207) D0 D3 D11'))
for err_loc in explained_errs[0].circuit_error_locations:
    if low_weight_failure_analysis_config['print_level'] >= 2:
        print(err_loc)


In [ ]:
explained_errs = low_weight_failure_analysis_circ.explain_detector_error_model_errors(dem_filter=stim.DetectorErrorModel('error(0.0123207) D1 D4 D12'))
for err_loc in explained_errs[0].circuit_error_locations:
    if low_weight_failure_analysis_config['print_level'] >= 2:
        print(err_loc)


In [ ]:
explained_errs = low_weight_failure_analysis_circ.explain_detector_error_model_errors(dem_filter=stim.DetectorErrorModel('error(0.0214087) D0 D3 D4'))
for err_loc in explained_errs[0].circuit_error_locations:
    if low_weight_failure_analysis_config['print_level'] >= 2:
        print(err_loc)


In [ ]:
explained_errs = low_weight_failure_analysis_circ.explain_detector_error_model_errors(dem_filter=stim.DetectorErrorModel('error(0.0123207) D1 D11 D12'))
for err_loc in explained_errs[0].circuit_error_locations:
    if low_weight_failure_analysis_config['print_level'] >= 2:
        print(err_loc)


#### Hyperedge between layers -> CNOT

### Inspect cross-layer hyperedges

Examine the selected disagreement supports and circuit explanations around cross-layer faults. The examples show how a DEM error can couple several detectors and should not be read as a prevalence estimate.


In [ ]:
bfts_data = low_weight_failure_analysis_weight2_data[(low_weight_failure_analysis_weight2_data['pred_b'] != low_weight_failure_analysis_weight2_data['true']) & (low_weight_failure_analysis_weight2_data['pred_t'] == low_weight_failure_analysis_weight2_data['true'])]
bs_data = low_weight_failure_analysis_weight2_data[(low_weight_failure_analysis_weight2_data['pred_b'] == low_weight_failure_analysis_weight2_data['true']) & (low_weight_failure_analysis_weight2_data['pred_t'] == low_weight_failure_analysis_weight2_data['true'])]


In [ ]:
bfts_data


In [ ]:
low_weight_failure_analysis_builder = CircuitBuilder(low_weight_failure_analysis_code)
low_weight_failure_analysis_circ = low_weight_failure_analysis_builder.get_circuit_level_circuit(ErrorRate(p=0.023101, p_meas=0.0), d_rounds=1)
low_weight_failure_analysis_dem = low_weight_failure_analysis_circ.detector_error_model()
TG, low_weight_failure_analysis_ = build_tanner_graph_from_dem(low_weight_failure_analysis_dem)


In [ ]:
low_weight_failure_analysis_ps = [d['p'] for n, d in TG.nodes(data=True) if d.get('kind') == 'error']
plt.plot(low_weight_failure_analysis_ps, marker='o')


In [ ]:
low_weight_failure_analysis_builder = CircuitBuilder(low_weight_failure_analysis_code)
low_weight_failure_analysis_circ = low_weight_failure_analysis_builder.get_circuit_level_circuit(ErrorRate(p=0.1, p_meas=0.0), d_rounds=1)
low_weight_failure_analysis_dem = low_weight_failure_analysis_circ.detector_error_model()
TG, low_weight_failure_analysis_ = build_tanner_graph_from_dem(low_weight_failure_analysis_dem)


In [ ]:
low_weight_failure_analysis_ps = [d['p'] for n, d in TG.nodes(data=True) if d.get('kind') == 'error']
plt.plot(low_weight_failure_analysis_ps, marker='o')


In [ ]:
visualize_det_on_tanner_graph(TG, '11010000000010', other_events={
    ('error', 'green', 'circle'): '010000010000000000000000000000000',
    ('bposd_corr', 'orange', 'diamond'): '100000000000010000000000000000010',
})


In [ ]:
bs_data[bs_data['det'] == '11010000000010']


In [ ]:
visualize_det_on_tanner_graph(TG, '11010000000010', other_events={('error', 'green', 'circle'): '010000010000000000000000000000000'})


In [ ]:
visualize_det_on_tanner_graph(TG, '11010000000010', other_events={('error', 'green', 'circle'): '001000000100000000000000000000000'})


## Weight-3

### Repeat acquisition at weight 3

Construct and decode the weight-3 support population, then compare the two disagreement directions. The following feature and belief panels use this weight-specific table.


In [ ]:
low_weight_failure_analysis_weight = 3


In [ ]:
low_weight_failure_analysis_dem = CircuitBuilder(low_weight_failure_analysis_code).get_circuit_level_circuit(ErrorRate(p=0.01, p_meas=0.0), d_rounds=1).detector_error_model()
low_weight_failure_analysis_demMatrices = low_weight_failure_analysis_detector_error_model_to_check_matrices(low_weight_failure_analysis_dem, allow_undecomposed_hyperedges=True)
low_weight_failure_analysis_h = low_weight_failure_analysis_demMatrices.check_matrix
low_weight_failure_analysis_l = low_weight_failure_analysis_demMatrices.observables_matrix
low_weight_failure_analysis_errs, low_weight_failure_analysis_dets, low_weight_failure_analysis_obs = sample_on_error_vec(low_weight_failure_analysis_num_errs, low_weight_failure_analysis_weight, low_weight_failure_analysis_h, low_weight_failure_analysis_l)
low_weight_failure_analysis_dets_packed = low_weight_failure_analysis_pack_bits_little(low_weight_failure_analysis_dets)
low_weight_failure_analysis_obs_packed = low_weight_failure_analysis_pack_bits_little(low_weight_failure_analysis_obs)


In [ ]:
from tqdm.auto import tqdm

low_weight_failure_analysis_recs = []
low_weight_failure_analysis_builder = CircuitBuilder(low_weight_failure_analysis_code)
low_weight_failure_analysis_tag_builder = CircuitBuilder(low_weight_failure_analysis_code, need_tags=True)

for low_weight_failure_analysis_p in tqdm(low_weight_failure_analysis_p_values, desc='Scientific sweep', disable=low_weight_failure_analysis_config['print_level'] == 1):
    low_weight_failure_analysis_error_rate = ErrorRate(p=float(low_weight_failure_analysis_p), p_meas=0.0)
    low_weight_failure_analysis_circ = low_weight_failure_analysis_builder.get_circuit_level_circuit(low_weight_failure_analysis_error_rate, d_rounds=1)
    low_weight_failure_analysis_tag_circ = low_weight_failure_analysis_tag_builder.get_circuit_level_circuit(low_weight_failure_analysis_error_rate, d_rounds=1)
    low_weight_failure_analysis_dem = low_weight_failure_analysis_circ.detector_error_model()
    low_weight_failure_analysis_g, low_weight_failure_analysis_ = build_tanner_graph_from_dem(low_weight_failure_analysis_dem)

    low_weight_failure_analysis_compiled_b = BPOSD(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, low_weight_failure_analysis_max_iter).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)
    low_weight_failure_analysis_compiled_t = Tesseract(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)

    low_weight_failure_analysis_corr_b, low_weight_failure_analysis_pred_b = low_weight_failure_analysis_compiled_b.decode_with_correction_shots_bit_packed(bit_packed_detection_event_data=low_weight_failure_analysis_dets_packed)
    low_weight_failure_analysis_corr_b = unpack_bits_little(low_weight_failure_analysis_corr_b, low_weight_failure_analysis_num_errs)
    low_weight_failure_analysis_pred_b = unpack_bits_little(low_weight_failure_analysis_pred_b, low_weight_failure_analysis_num_obs)

    low_weight_failure_analysis_corr_t = np.zeros_like(low_weight_failure_analysis_corr_b)
    for low_weight_failure_analysis_i, low_weight_failure_analysis_det in enumerate(low_weight_failure_analysis_dets):
        low_weight_failure_analysis_corr_t[low_weight_failure_analysis_i, :] = id2bits(low_weight_failure_analysis_compiled_t.decoder.decode_to_errors(low_weight_failure_analysis_det), low_weight_failure_analysis_num_errs)

    low_weight_failure_analysis_pred_t = low_weight_failure_analysis_corr_t @ low_weight_failure_analysis_l.T % 2

    low_weight_failure_analysis_recs.extend(
        {
            'p': low_weight_failure_analysis_p,
            'err': bits2str(e),
            'err_cross_time': error_cross_time(get_error_time(low_weight_failure_analysis_dem, bits2str(e))),
            'err_llr_weight': error_llr_weight(low_weight_failure_analysis_dem, bits2str(e)),
            'err_diameter': error_diameter(low_weight_failure_analysis_g, bits2str(e)),
            'err_most_common_src': [most_occurring_circ_error(dem_error_to_explanation(low_weight_failure_analysis_tag_circ, low_weight_failure_analysis_dem), err_idx) for err_idx in bits2id(e)],
            'det': bits2str(d),
            'corr_b': bits2str(cb),
            'corr_b_cross_time': error_cross_time(get_error_time(low_weight_failure_analysis_dem, bits2str(cb))),
            'corr_b_llr_weight': error_llr_weight(low_weight_failure_analysis_dem, bits2str(cb)),
            'corr_b_diameter': error_diameter(low_weight_failure_analysis_g, bits2str(cb)),
            'bp_converged': low_weight_failure_analysis_compiled_b.decoder.converge(),
            'corr_t': bits2str(ct),
            'corr_t_cross_time': error_cross_time(get_error_time(low_weight_failure_analysis_dem, bits2str(ct))),
            'corr_t_llr_weight': error_llr_weight(low_weight_failure_analysis_dem, bits2str(ct)),
            'corr_t_diameter': error_diameter(low_weight_failure_analysis_g, bits2str(ct)),
            'pred_b': bits2str(pb),
            'pred_t': bits2str(pt),
            'true': bits2str(ob),
        }
        for (e, d, cb, ct, pb, pt, ob) in zip(
            low_weight_failure_analysis_errs,
            low_weight_failure_analysis_dets,
            low_weight_failure_analysis_corr_b,
            low_weight_failure_analysis_corr_t,
            low_weight_failure_analysis_pred_b,
            low_weight_failure_analysis_pred_t,
            low_weight_failure_analysis_obs,
        )
    )

weight3_data = pd.DataFrame(low_weight_failure_analysis_recs)
low_weight_failure_analysis_save_path = low_weight_failure_analysis_data_paths.output('weight__low_weight_failure_analysis_weight__data_csv', low_weight_failure_analysis_weight=low_weight_failure_analysis_weight)

weight3_data.to_csv(low_weight_failure_analysis_save_path, index=False)
if low_weight_failure_analysis_config['print_level'] >= 2:
    print(f'saved: {low_weight_failure_analysis_save_path}')


In [ ]:
low_weight_failure_analysis_types = {
    'p': float,
    'err': str,
    'det': str,
    'corr_b': str,
    'corr_t': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

weight3_data = pd.read_csv(low_weight_failure_analysis_data_paths.output('weight__low_weight_failure_analysis_weight__data_csv', low_weight_failure_analysis_weight=low_weight_failure_analysis_weight), dtype=low_weight_failure_analysis_types)


In [ ]:
low_weight_failure_analysis_mask = (weight3_data['pred_b'] == weight3_data['true']) & (weight3_data['pred_t'] != weight3_data['true'])
low_weight_failure_analysis_filtered_data = weight3_data[low_weight_failure_analysis_mask]
low_weight_failure_analysis_filtered_data


In [ ]:
low_weight_failure_analysis_filtered_data[low_weight_failure_analysis_filtered_data['err_cross_time'] == False]


In [ ]:
low_weight_failure_analysis_mask = (weight3_data['pred_b'] != weight3_data['true']) & (weight3_data['pred_t'] == weight3_data['true'])
low_weight_failure_analysis_filtered_data = weight3_data[low_weight_failure_analysis_mask]
low_weight_failure_analysis_filtered_data


In [ ]:
low_weight_failure_analysis_filtered_data['bp_converged'].value_counts()


In [ ]:
low_weight_failure_analysis_filtered_data[low_weight_failure_analysis_filtered_data['err_cross_time'] == False]


In [ ]:
row = low_weight_failure_analysis_filtered_data.loc[11220]
err_ids = str2id(row['err'], low_weight_failure_analysis_dem.num_detectors)
corr_b_ids = str2id(row['corr_b'], low_weight_failure_analysis_dem.num_detectors)
(err_ids, corr_b_ids)


In [ ]:
low_weight_failure_analysis_circ = low_weight_failure_analysis_builder.get_circuit_level_circuit(ErrorRate(p=float(row['p']), p_meas=0.0), d_rounds=1)
low_weight_failure_analysis_dem = low_weight_failure_analysis_circ.detector_error_model()
TG, low_weight_failure_analysis_ = build_tanner_graph_from_dem(low_weight_failure_analysis_dem)


In [ ]:
visualize_det_on_tanner_graph(TG, row['det'], other_events={('err', 'green', 'circle'): row['err'], ('corr_b', 'orange', 'circle'): row['corr_b']})


the graph is asymmetric (notice the hyperedge in the middle and at the right) -> failure errors are not symmetric

### LLR Weight and Diameter

In [ ]:
low_weight_failure_analysis_fig, low_weight_failure_analysis_ax = plt.subplots(2, 1, figsize=(18, 10))
low_weight_failure_analysis_ax = low_weight_failure_analysis_ax.flatten()
low_weight_failure_analysis_df = low_weight_failure_analysis_filtered_data[['err_llr_weight', 'corr_b_llr_weight', 'corr_t_llr_weight']].copy().reset_index(drop=True)

low_weight_failure_analysis_df.plot(
    figsize=(12, 5),
    marker='o',
    ax=low_weight_failure_analysis_ax[0],
    xlabel='case index',
    ylabel='LLR weight',
    title='LLR weights for each case',
)
low_weight_failure_analysis_df = low_weight_failure_analysis_filtered_data[['err_diameter', 'corr_b_diameter', 'corr_t_diameter']].copy().reset_index(drop=True)
low_weight_failure_analysis_df.plot(
    figsize=(12, 5),
    marker='o',
    ax=low_weight_failure_analysis_ax[1],
    xlabel='case index',
    ylabel='Diameter',
    title='Diameters for each case',
)
plt.tight_layout()

plt.show()


### Belief Tacking

In [ ]:
from utils.belief_tracking import analyze_belief_history, plot_belief_analysis


In [ ]:
low_weight_failure_analysis_filtered_data.groupby(['det', 'err']).value_counts()


### Trace a selected weight-3 failure

Reconstruct the recorded input at its saved p and inspect the resulting belief/hard-decision history. The case is illustrative; its dynamics do not summarize all weight-3 failures.


In [ ]:
row = low_weight_failure_analysis_filtered_data[(low_weight_failure_analysis_filtered_data['det'] == '11111000100110') & (low_weight_failure_analysis_filtered_data['err'] == '010001000000000000000000000100000')].iloc[-1]
low_weight_failure_analysis_error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
low_weight_failure_analysis_builder = CircuitBuilder(low_weight_failure_analysis_code)
low_weight_failure_analysis_circ = low_weight_failure_analysis_builder.get_circuit_level_circuit(low_weight_failure_analysis_error_rate, d_rounds=1)
low_weight_failure_analysis_dem = low_weight_failure_analysis_circ.detector_error_model()
low_weight_failure_analysis_compiled_b = CythonBPOSD(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, low_weight_failure_analysis_max_iter).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)
low_weight_failure_analysis_compiled_t = Tesseract(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)


In [ ]:
low_weight_failure_analysis_pred_b = low_weight_failure_analysis_compiled_b.decoder.decode(str2bits(row['det']))
low_weight_failure_analysis_belief_history = low_weight_failure_analysis_compiled_b.decoder.belief_history.copy()
low_weight_failure_analysis_pred_t = low_weight_failure_analysis_compiled_t.decoder.decode(str2bits(row['det']))
low_weight_failure_analysis_info_b = {
    'pred_b': low_weight_failure_analysis_pred_b,
    'belief_history_shape': low_weight_failure_analysis_belief_history.shape,
    'is_bp_converged': low_weight_failure_analysis_compiled_b.decoder.is_bp_converged,
    'is_osd_satisfied': low_weight_failure_analysis_compiled_b.decoder.is_osd_satisfied,
}
low_weight_failure_analysis_info_t = {'pred_t': low_weight_failure_analysis_pred_t.astype(int)}

if low_weight_failure_analysis_config['print_level'] >= 2:
    print('BPOSD info:', low_weight_failure_analysis_info_b)
if low_weight_failure_analysis_config['print_level'] >= 2:
    print('Tesseract info:', low_weight_failure_analysis_info_t)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

prob_history = np.asarray(list(map(lambda x: llr2p(x), low_weight_failure_analysis_belief_history)))
num_steps, num_nodes = prob_history.shape
time_steps = np.arange(num_steps)
node_indices = np.arange(num_nodes)
X, Y = np.meshgrid(node_indices, time_steps)

low_weight_failure_analysis_fig = plt.figure(figsize=(10, 7))
low_weight_failure_analysis_ax = low_weight_failure_analysis_fig.add_subplot(111, projection='3d')
surf = low_weight_failure_analysis_ax.plot_surface(X, Y, prob_history, cmap='viridis', edgecolor='none', alpha=0.9)
low_weight_failure_analysis_ax.set_xlabel('Node index')
low_weight_failure_analysis_ax.set_ylabel('Time step')
low_weight_failure_analysis_ax.set_zlabel('Probability')
low_weight_failure_analysis_ax.set_title('Probability evolution over nodes and time')
low_weight_failure_analysis_fig.colorbar(surf, ax=low_weight_failure_analysis_ax, shrink=0.6, label='Probability')

plt.show()


In [ ]:
summary_df, variable_df, info = analyze_belief_history(low_weight_failure_analysis_belief_history, low_weight_failure_analysis_max_iter, low_weight_failure_analysis_compiled_b.decoder.H, str2bits(row['det']))
summary_df


In [ ]:
low_weight_failure_analysis_fig, low_weight_failure_analysis_ax = plot_belief_analysis(info, variable_df)


In [ ]:
hard_history = (low_weight_failure_analysis_belief_history < 0).astype(np.uint8)
low_weight_failure_analysis_fig, low_weight_failure_analysis_ax = plt.subplots(3, 1, figsize=(12, 12))
plot_belief_tracking(low_weight_failure_analysis_belief_history, low_weight_failure_analysis_ax[0])
plot_history_heatmap(low_weight_failure_analysis_belief_history, kind='belief', fig=low_weight_failure_analysis_fig, ax=low_weight_failure_analysis_ax[1])
plot_history_heatmap(hard_history, kind='hard decision', fig=low_weight_failure_analysis_fig, ax=low_weight_failure_analysis_ax[2])


In [ ]:
TG, low_weight_failure_analysis_ = build_tanner_graph_from_dem(low_weight_failure_analysis_dem)
hard_history = (low_weight_failure_analysis_belief_history < 0).astype(np.uint8)
hard_decisions_list = list(map(lambda x: ''.join(map(str, x)), hard_history))
ani = animate_hard_decisions_on_graph(TG, row['det'], hard_decisions_list, err_events=row['err'])

ani.show()


## Weight-4

### Repeat acquisition at weight 4

Generate the weight-4 support table and select the disagreement subset for the following diagnostic plots. A higher weight changes the tested population, so retain its own denominator.


In [ ]:
low_weight_failure_analysis_weight = 4


In [ ]:
low_weight_failure_analysis_dem = CircuitBuilder(low_weight_failure_analysis_code).get_circuit_level_circuit(ErrorRate(p=0.01, p_meas=0.0), d_rounds=1).detector_error_model()
low_weight_failure_analysis_demMatrices = low_weight_failure_analysis_detector_error_model_to_check_matrices(low_weight_failure_analysis_dem, allow_undecomposed_hyperedges=True)
low_weight_failure_analysis_h = low_weight_failure_analysis_demMatrices.check_matrix
low_weight_failure_analysis_l = low_weight_failure_analysis_demMatrices.observables_matrix
low_weight_failure_analysis_errs, low_weight_failure_analysis_dets, low_weight_failure_analysis_obs = sample_on_error_vec(low_weight_failure_analysis_num_errs, low_weight_failure_analysis_weight, low_weight_failure_analysis_h, low_weight_failure_analysis_l)
low_weight_failure_analysis_dets_packed = low_weight_failure_analysis_pack_bits_little(low_weight_failure_analysis_dets)
low_weight_failure_analysis_obs_packed = low_weight_failure_analysis_pack_bits_little(low_weight_failure_analysis_obs)


In [ ]:
from tqdm.auto import tqdm

low_weight_failure_analysis_recs = []
low_weight_failure_analysis_builder = CircuitBuilder(low_weight_failure_analysis_code)
low_weight_failure_analysis_tag_builder = CircuitBuilder(low_weight_failure_analysis_code, need_tags=True)

for low_weight_failure_analysis_p in tqdm(low_weight_failure_analysis_p_values, desc='Scientific sweep', disable=low_weight_failure_analysis_config['print_level'] == 1):
    low_weight_failure_analysis_error_rate = ErrorRate(p=float(low_weight_failure_analysis_p), p_meas=0.0)
    low_weight_failure_analysis_circ = low_weight_failure_analysis_builder.get_circuit_level_circuit(low_weight_failure_analysis_error_rate, d_rounds=1)
    low_weight_failure_analysis_tag_circ = low_weight_failure_analysis_tag_builder.get_circuit_level_circuit(low_weight_failure_analysis_error_rate, d_rounds=1)
    low_weight_failure_analysis_dem = low_weight_failure_analysis_circ.detector_error_model()
    low_weight_failure_analysis_g, low_weight_failure_analysis_ = build_tanner_graph_from_dem(low_weight_failure_analysis_dem)

    low_weight_failure_analysis_compiled_b = BPOSD(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, low_weight_failure_analysis_max_iter).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)
    low_weight_failure_analysis_compiled_t = Tesseract(low_weight_failure_analysis_code, low_weight_failure_analysis_error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=low_weight_failure_analysis_dem)

    low_weight_failure_analysis_corr_b, low_weight_failure_analysis_pred_b = low_weight_failure_analysis_compiled_b.decode_with_correction_shots_bit_packed(bit_packed_detection_event_data=low_weight_failure_analysis_dets_packed)
    low_weight_failure_analysis_corr_b = unpack_bits_little(low_weight_failure_analysis_corr_b, low_weight_failure_analysis_num_errs)
    low_weight_failure_analysis_pred_b = unpack_bits_little(low_weight_failure_analysis_pred_b, low_weight_failure_analysis_num_obs)

    low_weight_failure_analysis_corr_t = np.zeros_like(low_weight_failure_analysis_corr_b)
    for low_weight_failure_analysis_i, low_weight_failure_analysis_det in enumerate(low_weight_failure_analysis_dets):
        low_weight_failure_analysis_corr_t[low_weight_failure_analysis_i, :] = id2bits(low_weight_failure_analysis_compiled_t.decoder.decode_to_errors(low_weight_failure_analysis_det), low_weight_failure_analysis_num_errs)

    low_weight_failure_analysis_pred_t = low_weight_failure_analysis_corr_t @ low_weight_failure_analysis_l.T % 2

    low_weight_failure_analysis_recs.extend(
        {
            'p': low_weight_failure_analysis_p,
            'err': bits2str(e),
            'err_cross_time': error_cross_time(get_error_time(low_weight_failure_analysis_dem, bits2str(e))),
            'err_llr_weight': error_llr_weight(low_weight_failure_analysis_dem, bits2str(e)),
            'err_diameter': error_diameter(low_weight_failure_analysis_g, bits2str(e)),
            'err_most_common_src': [most_occurring_circ_error(dem_error_to_explanation(low_weight_failure_analysis_tag_circ, low_weight_failure_analysis_dem), err_idx) for err_idx in bits2id(e)],
            'det': bits2str(d),
            'corr_b': bits2str(cb),
            'corr_b_cross_time': error_cross_time(get_error_time(low_weight_failure_analysis_dem, bits2str(cb))),
            'corr_b_llr_weight': error_llr_weight(low_weight_failure_analysis_dem, bits2str(cb)),
            'corr_b_diameter': error_diameter(low_weight_failure_analysis_g, bits2str(cb)),
            'bp_converged': low_weight_failure_analysis_compiled_b.decoder.converge(),
            'corr_t': bits2str(ct),
            'corr_t_cross_time': error_cross_time(get_error_time(low_weight_failure_analysis_dem, bits2str(ct))),
            'corr_t_llr_weight': error_llr_weight(low_weight_failure_analysis_dem, bits2str(ct)),
            'corr_t_diameter': error_diameter(low_weight_failure_analysis_g, bits2str(ct)),
            'pred_b': bits2str(pb),
            'pred_t': bits2str(pt),
            'true': bits2str(ob),
        }
        for (e, d, cb, ct, pb, pt, ob) in zip(
            low_weight_failure_analysis_errs,
            low_weight_failure_analysis_dets,
            low_weight_failure_analysis_corr_b,
            low_weight_failure_analysis_corr_t,
            low_weight_failure_analysis_pred_b,
            low_weight_failure_analysis_pred_t,
            low_weight_failure_analysis_obs,
        )
    )

low_weight_failure_analysis_weight4_data = pd.DataFrame(low_weight_failure_analysis_recs)
low_weight_failure_analysis_save_path = low_weight_failure_analysis_data_paths.output('weight__low_weight_failure_analysis_weight__data_csv', low_weight_failure_analysis_weight=low_weight_failure_analysis_weight)

low_weight_failure_analysis_weight4_data.to_csv(low_weight_failure_analysis_save_path, index=False)
if low_weight_failure_analysis_config['print_level'] >= 2:
    print(f'saved: {low_weight_failure_analysis_save_path}')


In [ ]:
low_weight_failure_analysis_types = {
    'p': float,
    'err': str,
    'det': str,
    'corr_b': str,
    'corr_t': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

low_weight_failure_analysis_weight4_data = pd.read_csv(low_weight_failure_analysis_data_paths.output('weight__low_weight_failure_analysis_weight__data_csv', low_weight_failure_analysis_weight=low_weight_failure_analysis_weight), dtype=low_weight_failure_analysis_types)


In [ ]:
low_weight_failure_analysis_mask = (low_weight_failure_analysis_weight4_data['pred_b'] != low_weight_failure_analysis_weight4_data['true']) & (low_weight_failure_analysis_weight4_data['pred_t'] == low_weight_failure_analysis_weight4_data['true'])
low_weight_failure_analysis_filtered_data = low_weight_failure_analysis_weight4_data[low_weight_failure_analysis_mask]
low_weight_failure_analysis_filtered_data


In [ ]:
low_weight_failure_analysis_filtered_data['bp_converged'].value_counts()


### LLR Weight and Diameter

In [ ]:
low_weight_failure_analysis_fig, low_weight_failure_analysis_ax = plt.subplots(2, 1, figsize=(18, 10))
low_weight_failure_analysis_ax = low_weight_failure_analysis_ax.flatten()
low_weight_failure_analysis_df = low_weight_failure_analysis_filtered_data[['err_llr_weight', 'corr_b_llr_weight', 'corr_t_llr_weight']].copy().reset_index(drop=True)

low_weight_failure_analysis_df.plot(
    figsize=(12, 5),
    marker='o',
    ax=low_weight_failure_analysis_ax[0],
    xlabel='case index',
    ylabel='LLR weight',
    title='LLR weights for each case',
)
low_weight_failure_analysis_df = low_weight_failure_analysis_filtered_data[['err_diameter', 'corr_b_diameter', 'corr_t_diameter']].copy().reset_index(drop=True)
low_weight_failure_analysis_df.plot(
    figsize=(12, 5),
    marker='o',
    ax=low_weight_failure_analysis_ax[1],
    xlabel='case index',
    ylabel='Diameter',
    title='Diameters for each case',
)
plt.tight_layout()

plt.show()


### Belief Tacking

In [ ]:
from utils.belief_tracking import analyze_belief_history, plot_belief_analysis


In [ ]:
low_weight_failure_analysis_filtered_data.groupby(['det', 'err']).value_counts()


# Error Weight Prob
Configuration: `config`. Figures remain local; datasets are under `experiment/data/`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
error_weight_prob_experiment_dir = ROOT / 'experiment/research/low_weight_failures'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

error_weight_prob_data_paths = ExperimentPaths(ROOT, error_weight_prob_experiment_dir / 'config/paths.json')

import json

error_weight_prob_config = json.loads((error_weight_prob_experiment_dir / 'config/runtime.json').read_text())
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.


In [ ]:
import multiprocessing
import sys
import os
from pathlib import Path
from typing import Literal
import numpy as np
import pandas as pd
import sinter
import stim
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product, combinations

error_weight_prob_data_dir = error_weight_prob_data_paths.run()
error_weight_prob_data_dir.mkdir(parents=True, exist_ok=True)

from utils.misc import *

%load_ext autoreload
%autoreload 2


In [ ]:
import networkx as nx
from beliefmatching import detector_error_model_to_check_matrices as error_weight_prob_detector_error_model_to_check_matrices
from utils.sample import sample_on_error_vec
from utils.misc import pack_bits_little as error_weight_prob_pack_bits_little
from utils.sample import packed_rows_equal
from utils.graph_struct import build_tanner_graph_from_dem, error_llr_weight, get_error_time, error_cross_time, error_diameter, dem_error_to_explanation, most_occurring_circ_error
from utils.graph_vis import visualize_graph_plotly, visualize_det_on_tanner_graph, animate_hard_decisions_on_graph
from utils.belief_tracking import plot_belief_tracking


In [ ]:
from codes.ca_code import CA_5_2_3, CA_9_2_5, TriangularCode
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders import BPOSD
from src.decoders import Tesseract
from src.decoders import CythonBPOSD


In [ ]:
error_weight_prob_ca_codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5(), 'CA_14_2_9': TriangularCode(3)}

from experiment.utils import decoder_setup

def error_weight_prob_build_decoder_pair(code, p):
    return decoder_setup.build_decoder_pair(code, p, bposd_type=BPOSD, tesseract_type=Tesseract)


In [ ]:
error_weight_prob_weight_dir = error_weight_prob_data_paths.output('error_weight')
os.makedirs(error_weight_prob_weight_dir, exist_ok=True)
error_weight_prob_num_errs = 33
error_weight_prob_num_dets = 14
error_weight_prob_num_obs = 2
error_weight_prob_max_iter = 60
error_weight_prob_code = error_weight_prob_ca_codes['CA_9_2_5']


## Collect Errors

### Prepare the error-weight probability study

Read the configured sampling inputs and reconstruct the selected code for the next weight-dependent calculation. This section has its own decoder context rather than inheriting a previous example's matrix shape.


In [ ]:
error_weight_prob_max_shots = 1000000000
error_weight_prob_d_rounds = 1
error_weight_prob_noise_model = 'circuit_level'
CASES_DIR = error_weight_prob_data_paths.input('failure_cases')


In [ ]:
error_weight_prob_types = {
    'detector_events': str,
    'error_events': str,
    'true_observables': str,
    'bposd_pred_observables': str,
    'tesseract_pred_observables': str,
}

error_weight_prob_bfts_cases = pd.read_csv(error_weight_prob_data_paths.input('bfts_cases_csv'), dtype=error_weight_prob_types)


In [ ]:
error_weight_prob_p_values = error_weight_prob_bfts_cases['p'].unique()
error_weight_prob_p_values


In [ ]:
error_weight_prob_code = error_weight_prob_ca_codes['CA_14_2_9']
error_weight_prob_weight = (error_weight_prob_code.d - 1) // 2


In [ ]:
error_weight_prob_dem = CircuitBuilder(error_weight_prob_code).get_circuit_level_circuit(ErrorRate(p=0.01, p_meas=0.0), d_rounds=1).detector_error_model()
error_weight_prob_demMatrices = error_weight_prob_detector_error_model_to_check_matrices(error_weight_prob_dem, allow_undecomposed_hyperedges=True)
error_weight_prob_h = error_weight_prob_demMatrices.check_matrix
error_weight_prob_l = error_weight_prob_demMatrices.observables_matrix
error_weight_prob_num_errs = error_weight_prob_h.shape[1]
error_weight_prob_num_dets = error_weight_prob_h.shape[0]
error_weight_prob_num_obs = error_weight_prob_l.shape[0]
error_weight_prob_errs, error_weight_prob_dets, error_weight_prob_obs = sample_on_error_vec(error_weight_prob_num_errs, error_weight_prob_weight, error_weight_prob_h, error_weight_prob_l)
error_weight_prob_dets_packed = error_weight_prob_pack_bits_little(error_weight_prob_dets)
error_weight_prob_obs_packed = error_weight_prob_pack_bits_little(error_weight_prob_obs)


In [ ]:
error_weight_prob_errs.shape


### Collect weight-dependent decoder outcomes

Run the configured collection and inspect the saved weight-specific disagreements. Interpret the results using the support selection and code active in this section.


In [ ]:
error_weight_prob_recs = []
error_weight_prob_builder = CircuitBuilder(error_weight_prob_code)
error_weight_prob_tag_builder = CircuitBuilder(error_weight_prob_code, need_tags=True)

for error_weight_prob_p in error_weight_prob_p_values:
    error_weight_prob_error_rate = ErrorRate(p=float(error_weight_prob_p), p_meas=0.0)
    error_weight_prob_circ = error_weight_prob_builder.get_circuit_level_circuit(error_weight_prob_error_rate, d_rounds=1)
    error_weight_prob_dem = error_weight_prob_circ.detector_error_model()
    error_weight_prob_g, error_weight_prob_ = build_tanner_graph_from_dem(error_weight_prob_dem)
    error_weight_prob_compiled_b = BPOSD(error_weight_prob_code, error_weight_prob_error_rate.p, error_weight_prob_max_iter).compile_decoder_for_dem(dem=error_weight_prob_dem)
    error_weight_prob_compiled_t = Tesseract(error_weight_prob_code, error_weight_prob_error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=error_weight_prob_dem)
    error_weight_prob_corr_b, error_weight_prob_pred_b = error_weight_prob_compiled_b.decode_with_correction_shots_bit_packed(bit_packed_detection_event_data=error_weight_prob_dets_packed)
    error_weight_prob_corr_b = unpack_bits_little(error_weight_prob_corr_b, error_weight_prob_num_errs)
    error_weight_prob_pred_b = unpack_bits_little(error_weight_prob_pred_b, error_weight_prob_num_obs)
    error_weight_prob_corr_t = np.zeros_like(error_weight_prob_corr_b)

    for error_weight_prob_i, error_weight_prob_det in enumerate(error_weight_prob_dets):
        error_weight_prob_corr_t[error_weight_prob_i, :] = id2bits(error_weight_prob_compiled_t.decoder.decode_to_errors(error_weight_prob_det), error_weight_prob_num_errs)
    error_weight_prob_pred_t = error_weight_prob_corr_t @ error_weight_prob_l.T % 2
    error_weight_prob_recs.extend(({
        'p': error_weight_prob_p,
        'err': bits2str(e),
        'err_llr_weight': error_llr_weight(error_weight_prob_dem, bits2str(e)),
        'det': bits2str(d),
        'corr_b': bits2str(cb),
        'corr_b_llr_weight': error_llr_weight(error_weight_prob_dem, bits2str(cb)),
        'bp_converged': error_weight_prob_compiled_b.decoder.converge(),
        'corr_t': bits2str(ct),
        'corr_t_llr_weight': error_llr_weight(error_weight_prob_dem, bits2str(ct)),
        'pred_b': bits2str(pb),
        'pred_t': bits2str(pt),
        'true': bits2str(ob),
    } for e, d, cb, ct, pb, pt, ob in zip(error_weight_prob_errs, error_weight_prob_dets, error_weight_prob_corr_b, error_weight_prob_corr_t, error_weight_prob_pred_b, error_weight_prob_pred_t, error_weight_prob_obs)))

error_weight_prob_weight4_data = pd.DataFrame(error_weight_prob_recs)
error_weight_prob_save_path = error_weight_prob_data_paths.output('d_parameter_1__weight__error_weight_prob_weight__data_csv', parameter_1=error_weight_prob_code.d, error_weight_prob_weight=error_weight_prob_weight)
error_weight_prob_weight4_data.to_csv(error_weight_prob_save_path, index=False)

if error_weight_prob_config['print_level'] >= 2:
    print(f'saved: {error_weight_prob_save_path}')


In [ ]:
error_weight_prob_types = {
    'p': float,
    'err': str,
    'det': str,
    'corr_b': str,
    'corr_t': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

error_weight_prob_weight4_data = pd.read_csv(error_weight_prob_data_paths.output('d_parameter_1__weight__error_weight_prob_weight__data_csv', parameter_1=error_weight_prob_code.d, error_weight_prob_weight=error_weight_prob_weight), dtype=error_weight_prob_types)


In [ ]:
error_weight_prob_mask = (error_weight_prob_weight4_data['pred_b'] != error_weight_prob_weight4_data['true']) & (error_weight_prob_weight4_data['pred_t'] == error_weight_prob_weight4_data['true'])
error_weight_prob_filtered_data = error_weight_prob_weight4_data[error_weight_prob_mask]
error_weight_prob_filtered_data


## Sweep Search for Threshold

prior threshold

## Prior and min-sum scaling: d=5 and d=9

This section owns its configuration and state. It compares all weight-2 DEM supports at both distances and a fixed-seed sample of weight-4 supports at distance 9. The same supports are retained across physical noise rates and scaling values. Rates are conditional on these uniformly weighted supports; they are not physical-noise LER estimates.


### Initialize the distance/scaling comparison

Load this section's configuration and independent d=5/d=9 contexts before running the sweep. Stable support IDs allow paired comparisons across p and alpha within each cohort.


In [ ]:
from IPython.display import Markdown, display
from experiment.utils import prior_scaling

prior_scaling_dir = ROOT / 'experiment/research/low_weight_failures'
prior_scaling_config = json.loads((prior_scaling_dir / 'config/prior_scaling.json').read_text())
prior_scaling_paths = ExperimentPaths(ROOT, prior_scaling_dir / 'config/prior_scaling_paths.json')

# print_level: 1 = silent, 2 = key progress, 3 = per-case progress and diagnostics.
# Distances are 5 and 9; the sampled cohort is drawn without replacement.
# All alpha values are positive; every p value is between 0 and 0.5.
prior_scaling.validate_config(prior_scaling_config)


In [ ]:
prior_scaling_summary, prior_scaling_boundaries = prior_scaling.run_sweep(prior_scaling_paths, prior_scaling_config)

display(prior_scaling_summary.head(), prior_scaling_boundaries.head())


### Failure fractions over p and alpha

BP-only uses the instrumented Cython min-sum implementation with canonical sequential accumulation. BPOSD uses the existing C++ wrapper and includes its OSD fallback. Tesseract is evaluated once per support and p, independently of alpha. A successful correction must satisfy the syndrome and predict the true logical observables.


In [ ]:
prior_scaling.plot_comparisons(prior_scaling_summary, prior_scaling_paths)


### Discrete alpha boundaries and non-monotonic behavior

The first failing alpha is a value on the configured grid, not an interpolated threshold. `alpha_robust` denotes the successful prefix before that first failure. If no tested alpha fails, the boundary is right-censored. Per-support boundary records remain in trace files; the table below aggregates their distributions and failure/recovery transitions.


In [ ]:
prior_scaling_transition_summary = prior_scaling_boundaries.groupby(['distance', 'weight', 'decoder', 'failure_transitions'], as_index=False)['samples'].sum()

display(prior_scaling_transition_summary)


### Available discrepancy examples

Each example is selected from its own distance, weight, p and alpha. No historical syndrome is reused across different PCM shapes. An empty discrepancy cohort is a valid result and produces an explicit explanation.


In [ ]:
for prior_scaling_case in prior_scaling_config['cases']:
    prior_scaling_distance = prior_scaling_case['distance']
    prior_scaling_weight = prior_scaling_case['weight']
    prior_scaling_example = prior_scaling.example_case(prior_scaling_summary, prior_scaling_paths, prior_scaling_distance, prior_scaling_weight)

    if prior_scaling_example is None:
        display(Markdown(f'No BPOSD-fail/Tesseract-success example was observed for d={prior_scaling_distance}, weight={prior_scaling_weight}.'))
    else:
        display(prior_scaling_example.to_frame('value'))
        prior_scaling.plot_example(prior_scaling_example, prior_scaling_config, prior_scaling_paths)


# Threshold Suppression
Configuration: `config`. Figures remain local; datasets are under `experiment/data/`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
threshold_suppression_experiment_dir = ROOT / 'experiment/research/low_weight_failures'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

threshold_suppression_data_paths = ExperimentPaths(ROOT, threshold_suppression_experiment_dir / 'config/paths.json')

import json

threshold_suppression_config = json.loads((threshold_suppression_experiment_dir / 'config/runtime.json').read_text())
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.


In [ ]:
import multiprocessing
import sys
import os
from pathlib import Path
from typing import Literal
import numpy as np
import pandas as pd
import sinter
import stim
import matplotlib.pyplot as plt
import seaborn as sns

threshold_suppression_data_dir = threshold_suppression_data_paths.run()
threshold_suppression_data_dir.mkdir(parents=True, exist_ok=True)

from utils.misc import *

%load_ext autoreload
%autoreload 2


In [ ]:
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders import BPOSD
from src.decoders import Tesseract


In [ ]:
threshold_suppression_ca_codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5()}

from experiment.utils import decoder_setup

def threshold_suppression_build_decoder_pair(code, p):
    return decoder_setup.build_decoder_pair(code, p, bposd_type=BPOSD, tesseract_type=Tesseract)


### Fixed-weight discrepancy helpers

Define uniform fixed-weight sampling and the paired discrepancy benchmark used by the final example. Its finite sampled-support fractions are distinct from an asymptotic decoding threshold.


In [ ]:
def sample_fixed_weight_uniform(num_errors: int, k: int, rng: np.random.Generator):
    """
    Uniformly sample a support S with |S| = k.
    """

    return rng.choice(num_errors, size=k, replace=False)

def syndrome_and_observable_from_support(H, L, support):
    """
    Given support indices of DEM errors, compute:
        syndrome = H[:, support] @ 1 mod 2
        obs      = L[:, support] @ 1 mod 2
    """
    syndrome = np.asarray(H[:, support].sum(axis=1)).ravel() % 2
    obs = np.asarray(L[:, support].sum(axis=1)).ravel() % 2

    return (syndrome.astype(np.bool_), obs.astype(np.bool_))


In [ ]:
def benchmark_fixed_weight_discrepancy(
    code,
    dem,
    weights=range(1, 8),
    shots_per_weight=1000,
    seed=0,
):
    from beliefmatching import detector_error_model_to_check_matrices

    demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
    H = demMatrices.check_matrix.toarray()
    L = demMatrices.observables_matrix.toarray()
    compiled_b = BPOSD(code, 0.01, 60).compile_decoder_for_dem(dem=dem)
    compiled_t = Tesseract(code, 0.01, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=dem)
    rng = np.random.default_rng(seed)
    num_errors = H.shape[1]
    records = []

    from tqdm.auto import tqdm

    for k in tqdm(weights, desc='Fixed-weight sampling', disable=threshold_suppression_config['print_level'] == 1):
        counts = {
            'both_success': 0,
            'both_fail': 0,
            'bposd_fail_tess_success': 0,
            'bposd_success_tess_fail': 0,
            'bposd_tess_disagree': 0,
        }

        for _ in tqdm(range(shots_per_weight), desc=f'weight={k}', leave=False, disable=threshold_suppression_config['print_level'] == 1):
            support = sample_fixed_weight_uniform(num_errors, k, rng)
            syndrome, true_obs = syndrome_and_observable_from_support(H, L, support)
            packed = np.packbits(syndrome.astype(np.uint8)[None, :], axis=1, bitorder='little')
            predicted = compiled_b.decode_shots_bit_packed(bit_packed_detection_event_data=packed)
            bposd_obs = np.unpackbits(predicted, axis=1, count=L.shape[0], bitorder='little')[0].astype(bool)
            packed = np.packbits(syndrome.astype(np.uint8)[None, :], axis=1, bitorder='little')
            predicted = compiled_t.decode_shots_bit_packed(bit_packed_detection_event_data=packed)
            tess_obs = np.unpackbits(predicted, axis=1, count=L.shape[0], bitorder='little')[0].astype(bool)
            bposd_success = np.array_equal(bposd_obs, true_obs)
            tess_success = np.array_equal(tess_obs, true_obs)

            if not np.array_equal(bposd_obs, tess_obs):
                counts['bposd_tess_disagree'] += 1
            if bposd_success and tess_success:
                counts['both_success'] += 1
            elif not bposd_success and tess_success:
                counts['bposd_fail_tess_success'] += 1
            elif bposd_success and (not tess_success):
                counts['bposd_success_tess_fail'] += 1
            else:
                counts['both_fail'] += 1

        row = {'weight': k, 'shots': shots_per_weight, 'seed': seed}
        for key, value in counts.items():
            row[key] = value / shots_per_weight
            row[key + '_count'] = value

        records.append(row)

    return records


### Run the final discrepancy comparison

Evaluate the explicitly selected code and weight settings using the helper above. Read paired logical outcomes together with their fixed seed and sampled-support count.


In [ ]:
threshold_suppression_code = threshold_suppression_ca_codes['CA_9_2_5']
threshold_suppression_dem = CircuitBuilder(threshold_suppression_code).get_circuit_level_circuit(ErrorRate(p=0.01, p_meas=0.0), d_rounds=1).detector_error_model()
records = benchmark_fixed_weight_discrepancy(threshold_suppression_code, threshold_suppression_dem, weights=range(1, 10), shots_per_weight=5000, seed=1234)
STATS = threshold_suppression_data_paths.stats()
STATS.mkdir(parents=True, exist_ok=True)

pd.DataFrame(records).to_csv(threshold_suppression_data_paths.output('fixed_weight_discrepancy_csv'), index=False)
if threshold_suppression_config['print_level'] >= 2:
    display(pd.DataFrame(records))
